# Libraries

In [ ]:
#!pip install -q torch==2.9.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip -q install "lm_eval[hf]"
!pip -q install --upgrade transformers
!pip -q install langdetect --break-system-packages #for If-eval Task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 3.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [3]:
MODEL_NAME       = "Llama-3.2-3B-Instruct-SparseGPT-25"
MODEL_PRETRAINED = "EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT"
SUB_FOLDER = "Models/25"
SEED             = 42

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Evaluations

**Configurations**

In [ ]:
# Task definition
TASKS = [
    # Math
     Task("gsm8k",        "gsm8k_cot",                 "math",                num_fewshot=8, batch_size=12,  max_gen_toks=1024),

    # Reasoning
     Task("arc_challenge","arc_challenge",              "reasoning",           batch_size=16),

    # Instruction Following
     Task("ifeval",       "ifeval",                    "instruction_following",batch_size=14,  max_gen_toks=1024),

    # Perplexity
     Task("wikitext",     "wikitext",                  "perplexity",          batch_size=1,  apply_chat_template=False),

    # Knowledge
     Task("mmlu",         "mmlu",                      "knowledge",           num_fewshot=5, batch_size=2,  limit=1400),
]

# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"subfolder={SUB_FOLDER}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
    "trust_remote_code=True",
    #"enable_thinking=False",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [9]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "WARNING",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)

    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[MATH] math-500


2026-04-07:15:23:14 INFO     [config.evaluate_config:301] Using default fewshot_as_multiturn=True.
2026-04-07:15:23:20 INFO     [_cli.run:376] Selected Tasks: ['minerva_math500']
2026-04-07:15:23:22 WARNING  [evaluator:223] generation_kwargs: {'max_gen_toks': 2048} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
Map: 100%|██████████| 500/500 [00:00<00:00, 9859.62 examples/s]
2026-04-07:15:24:16 WARNING  [evaluator:333] Overwriting default num_fewshot of minerva_math500 from 4 to 0
2026-04-07:15:24:16 WARNING  [evaluator:490] Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running generate_until requests: 100%|██████████| 500/500 [2:00:21<00:00, 14.44s/it]  
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Llama-3.2-3B-Instruct-SparseGPT', 'subfolder': 'Models/25', 'dtype': 'float16', 'parallelize': True, 'max_length': 4096}), gen_kwargs: ({'max_gen_toks': 2048}), limit: None, num_fewshot: 0, batch_size: 8
|     Tasks     |Version|Filter|n-shot|  Metric   |   |Value|   |Stderr|
|---------------|------:|------|-----:|-----------|---|----:|---|-----:|
|minerva_math500|      3|none  |     0|exact_match|↑  |0.000|±  |0.0000|
|               |       |none  |     0|math_verify|↑  |0.356|±  |0.0214|

Done


# Save reports

In [10]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)